In [ ]:
%pip -q install langchain langchain-groq

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)
print(llm.invoke("Reply with only the word Ready").content)

Ready


In [ ]:
business = "Campus Print Shop"

SYSTEM = f"""
ROLE
You are the friendly assistant for {business}. Help students get print estimates.

SHOP FACTS
- Black-and-white printing costs ₹2 per printed side.
- Colour printing costs ₹10 per printed side.
- Spiral binding costs ₹30 per copy.
- Double-sided printing uses both sides of a sheet. The price per printed side stays the same.
- Only the shop owner can confirm a pickup time.

RULES
- Use the shop prices when calculating an estimate.
- Use earlier messages to remember the customer's name and order.
- If a detail needed for the estimate is missing, ask one short question.
- If the customer asks "what if", give a preview and keep the original order.
- If the customer asks about pickup, ask them to confirm with the shop owner.

REPLY STYLE
Use simple English. Write prices in rupees and use digits for quantities.
"""

print(SYSTEM)


ROLE
You are the friendly assistant for Campus Print Shop. Help students get print estimates.

SHOP FACTS
- Black-and-white printing costs ₹2 per printed side.
- Colour printing costs ₹10 per printed side.
- Spiral binding costs ₹30 per copy.
- Double-sided printing uses both sides of a sheet. The price per printed side stays the same.
- Only the shop owner can confirm a pickup time.

RULES
- Use the shop prices when calculating an estimate.
- Use earlier messages to remember the customer's name and order.
- If a detail needed for the estimate is missing, ask one short question.
- If the customer asks "what if", give a preview and keep the original order.
- If the customer asks about pickup, ask them to confirm with the shop owner.

REPLY STYLE
Use simple English. Write prices in rupees and use digits for quantities.



In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

def make_chain(system_prompt, temperature):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder("history"),
        ("human", "{question}")
    ])
    return prompt | llm.bind(temperature=temperature) | StrOutputParser()

chain = make_chain(SYSTEM, temperature=0.3)
history = []

def ask(question):
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Poojitha. I need one copy of 20 black-and-white pages without binding. What is the price?"))
print(ask("What is my name and how many pages did I ask for?"))

₹40 for 20 single‑sided black‑and‑white pages.  
Do you want double‑sided printing?
Your name is Poojitha, and you asked for 20 pages.


In [ ]:
%%writefile tests.csv
question,must_include
"What services does the Campus Print Shop offer?",binding
"How much does one colour printed side cost in rupees?",10
"What is the cost of one copy of 20 black-and-white pages without binding?",40
"How many sheets do I need for one double-sided copy of 20 pages?",10
"If I change the first page of my 20-page black-and-white document to colour, what will one copy cost without binding?",48
"What is the printing cost for one copy with 19 black-and-white pages and one colour page?",48
"How much is one spiral-bound copy with 19 black-and-white pages and one colour page?",78
"What is the total for two spiral-bound copies, each with 19 black-and-white pages and one colour page?",156
"My order is two spiral-bound copies, each with 19 black-and-white pages and one colour page. Can you summarise my order and give the total?",156
"Can you confirm that my print order will be ready for pickup at 4 PM today?",owner

Writing tests.csv


In [6]:
import csv

with open("tests.csv", encoding="utf-8", newline="") as file:
    tests = list(csv.DictReader(file))

def run_tests(test_chain, label):
    print(label)
    passed = 0
    answers = []

    for number, row in enumerate(tests, start=1):
        # Start each question with an empty chat history.
        answer = test_chain.invoke({
            "history": [],
            "question": row["question"]
        })
        answers.append(answer)

        expected = row["must_include"].strip()
        matched = expected.casefold() in answer.casefold()
        if matched:
            passed += 1

        print(f"\n{number}. {'PASS' if matched else 'FAIL'} | {row['question']}")
        print(f"Must include: {expected}")
        print(f"Bot reply: {answer}")

    print(f"\nPass rate: {passed}/{len(tests)} ({passed / len(tests):.0%})")
    return passed, answers

baseline, baseline_answers = run_tests(
    chain, "Original prompt | Temperature 0.3"
)

Original prompt | Temperature 0.3

1. PASS | What services does the Campus Print Shop offer?
Must include: binding
Bot reply: Campus Print Shop offers:

- Black‑and‑white printing (₹2 per printed side)  
- Colour printing (₹10 per printed side)  
- Double‑sided printing (uses both sides of a sheet, same price per side)  
- Spiral binding (₹30 per copy)  

Let me know what you’d like to print and I can give you an estimate.

2. PASS | How much does one colour printed side cost in rupees?
Must include: 10
Bot reply: One colour printed side costs ₹10.

3. PASS | What is the cost of one copy of 20 black-and-white pages without binding?
Must include: 40
Bot reply: The cost is ₹40.

4. PASS | How many sheets do I need for one double-sided copy of 20 pages?
Must include: 10
Bot reply: You’ll need 10 sheets for one double‑sided copy of 20 pages.

5. FAIL | If I change the first page of my 20-page black-and-white document to colour, what will one copy cost without binding?
Must include: 48
Bot 

In [7]:
SYSTEM_V2 = SYSTEM + """
For a bound order, add ₹30 to the printing cost of each copy,
then multiply that amount by the number of copies.
"""

improved_chain = make_chain(SYSTEM_V2, temperature=0.3)
improved, improved_answers = run_tests(improved_chain, "Updated prompt | Temperature 0.3")

if improved > baseline:
    change = "went up"
elif improved < baseline:
    change = "went down"
else:
    change = "stayed the same"

print(f"\nOriginal pass rate: {baseline}/{len(tests)} ({baseline / len(tests):.0%})")
print(f"Updated pass rate: {improved}/{len(tests)} ({improved / len(tests):.0%})")
print(f"The pass rate {change}.")

Updated prompt | Temperature 0.3

1. PASS | What services does the Campus Print Shop offer?
Must include: binding
Bot reply: Campus Print Shop offers:

* **Black‑and‑white printing** – ₹2 per printed side  
* **Colour printing** – ₹10 per printed side  
* **Double‑sided printing** – you get two sides on one sheet, price stays the same per side  
* **Spiral binding** – ₹30 per copy (added to the printing cost of each copy)  

If you want to pick up your order, you’ll need to confirm the time with the shop owner.

2. PASS | How much does one colour printed side cost in rupees?
Must include: 10
Bot reply: One colour printed side costs ₹10.

3. PASS | What is the cost of one copy of 20 black-and-white pages without binding?
Must include: 40
Bot reply: The cost is ₹40.

4. PASS | How many sheets do I need for one double-sided copy of 20 pages?
Must include: 10
Bot reply: You’ll need 10 sheets for one double‑sided copy of 20 pages.

5. PASS | If I change the first page of my 20-page black-an

In [8]:
warm_chain = make_chain(SYSTEM_V2, temperature=1.0)

score1, replies1 = run_tests(warm_chain, "Temperature 1.0 | Run 1")
score2, replies2 = run_tests(warm_chain, "Temperature 1.0 | Run 2")

different_replies = [
    number
    for number, (first, second) in enumerate(zip(replies1, replies2), start=1)
    if first.strip() != second.strip()
]

print("\nTemperature experiment results")
print(f"Run 1: {score1}/{len(tests)} ({score1 / len(tests):.0%})")
print(f"Run 2: {score2}/{len(tests)} ({score2 / len(tests):.0%})")
print("Did the pass rate change?", "Yes" if score1 != score2 else "No")
print("Questions with different reply text:", different_replies)


Temperature 1.0 | Run 1

1. PASS | What services does the Campus Print Shop offer?
Must include: binding
Bot reply: Campus Print Shop can help you with:

* **Black‑and‑white printing** – ₹2 for each printed side.  
* **Colour printing** – ₹10 for each printed side.  
* **Double‑sided printing** – you print on both sides of the same sheet, still at the same per‑side price.  
* **Spiral binding** – ₹30 for each copy that you want bound.  

(If you need to pick up your order, the shop owner will set the time.)

2. PASS | How much does one colour printed side cost in rupees?
Must include: 10
Bot reply: One colour printed side costs ₹10.

3. PASS | What is the cost of one copy of 20 black-and-white pages without binding?
Must include: 40
Bot reply: The cost is ₹40.  
(20 pages × ₹2 per page, no binding.)

4. PASS | How many sheets do I need for one double-sided copy of 20 pages?
Must include: 10
Bot reply: You’ll need 10 sheets.  
With double‑sided printing, each sheet gives 2 pages, so 20 

In [11]:
from IPython.display import Markdown, display

display(Markdown("""
## Agent or not? Three jobs in my Campus Print Shop

| Print shop job | Classification | Why it fits |
|---|---|---|
| Calculate the bill for two spiral-bound copies | Plain code | A Python function uses fixed prices. If one bound copy costs ₹78, two copies cost ₹156. |
| Explain single-sided and double-sided printing | One LLM call | The model can answer the student's question in one reply using the shop facts in its prompt. |
| Check whether an order can be ready by 4 PM | Agent | An agent could check the print queue and printer availability, then use both results to estimate whether the order can be ready on time. |
"""))


## Agent or not? Three jobs in my Campus Print Shop

| Print shop job | Classification | Why it fits |
|---|---|---|
| Calculate the bill for two spiral-bound copies | Plain code | A Python function uses fixed prices. If one bound copy costs ₹78, two copies cost ₹156. |
| Explain single-sided and double-sided printing | One LLM call | The model can answer the student's question in one reply using the shop facts in its prompt. |
| Check whether an order can be ready by 4 PM | Agent | An agent could check the print queue and printer availability, then use both results to estimate whether the order can be ready on time. |


In [12]:
from pathlib import Path

def show_score(score):
    return f"{score}/{len(tests)} ({score / len(tests):.0%})"

changed_questions = ", ".join(str(n) for n in different_replies) or "None"

readme = f"""# Day 3 – Campus Print Shop Assistant with LangChain

**BELLUM POOJITHA · 24B11AI035**
Drive Ready – AI Agents | Aditya University

## Task
Build a print shop assistant using LangChain, test it with 10 customer
questions, and measure how its replies change when I update the prompt
or change the temperature.

## How it works
The system prompt gives the assistant the shop rates: ₹2 per black-and-white
printed side, ₹10 per colour printed side, and ₹30 for spiral binding per copy.
A LangChain prompt template combines these instructions with the customer's
question and chat history. ChatGroq generates the reply, and StrOutputParser
returns it as plain text.

## How I tested it
The file `tests.csv` contains 10 customer questions. Each question has a
`must_include` word or number. Each test starts with an empty chat history
and passes when the expected text appears in the reply.

| Experiment | Temperature | Pass rate |
|---|---:|---:|
| Original prompt | 0.3 | {show_score(baseline)} |
| Updated prompt | 0.3 | {show_score(improved)} |
| Updated prompt, run 1 | 1.0 | {show_score(score1)} |
| Updated prompt, run 2 | 1.0 | {show_score(score2)} |

## What changed?
I added one billing rule: add the binding charge to each copy before
multiplying by the number of copies. After that change, the pass rate **{change}**.

At temperature 1.0, the two runs scored **{show_score(score1)}** and
**{show_score(score2)}**. Questions whose reply text differed: **{changed_questions}**.

## Agent or not?

| Print shop job | Classification | Reason |
|---|---|---|
| Calculate a bill | Plain code | A Python function can calculate the exact total from fixed rates and quantities. |
| Explain double-sided printing | One LLM call | The model can explain it in one reply using the shop facts. |
| Check whether an order can be ready by 4 PM | Agent | A future agent could check the print queue and printer availability before answering. |

## How to run
Open `Campus_Print_Shop_Agent_Day3.ipynb` in Google Colab. Enable notebook
access to `GROQ_API_KEY` in Colab Secrets, then run the cells from top
to bottom. The notebook creates `tests.csv` before running the tests.
"""

Path("README.md").write_text(readme, encoding="utf-8")
print("README.md is ready.")

README.md is ready.
